# Servir modelos ML con FastAPI

En este notebook aprenderemos a servir modelos de machine learning como APIs REST usando
**FastAPI**. Cubriremos desde la estructura basica hasta patrones avanzados como
optimizacion con ONNX Runtime y containerizacion con Docker.

**Objetivos:**
- Entender la brecha entre notebook y produccion
- Crear endpoints de prediccion con FastAPI
- Servir modelos sklearn y PyTorch
- Containerizar con Docker
- Testear el servicio
- Optimizar con ONNX Runtime

## De notebook a API

La brecha entre un modelo en un notebook y un modelo en produccion es enorme.
Un notebook es perfecto para experimentacion, pero en produccion necesitamos:

| Notebook | Produccion |
|----------|------------|
| Ejecucion manual | Servicio 24/7 |
| Un usuario | Miles de requests concurrentes |
| Input en variables | Input via HTTP/gRPC |
| Output en celdas | Output JSON estructurado |
| Sin manejo de errores | Error handling robusto |
| Sin validacion | Validacion de inputs estricta |
| Sin monitoreo | Logging, metricas, alertas |

### Por que FastAPI?
- **Rapido**: Basado en Starlette y uvicorn (async)
- **Validacion automatica**: Con Pydantic, valida inputs/outputs
- **Documentacion auto-generada**: Swagger UI en `/docs`
- **Type hints**: Codigo limpio y auto-documentado
- **Async nativo**: Maneja multiples requests eficientemente

## Estructura de un servicio ML

Un servicio ML bien organizado sigue una estructura clara que separa responsabilidades:
- **API layer**: Recibe requests, valida inputs, retorna responses
- **Model layer**: Carga y ejecuta el modelo
- **Preprocessing layer**: Transforma datos crudos al formato del modelo

In [ ]:
# Recommended project structure for an ML service

project_structure = """
ml-service/
|-- app/
|   |-- __init__.py
|   |-- main.py              # FastAPI app, endpoints
|   |-- models.py            # Pydantic request/response models
|   |-- ml/
|   |   |-- __init__.py
|   |   |-- predictor.py     # Model loading and prediction logic
|   |   |-- preprocessing.py # Feature engineering, transforms
|   |-- config.py            # Settings (model paths, thresholds)
|-- models/                  # Saved model artifacts
|   |-- model_v1.joblib
|   |-- model_v1.onnx
|-- tests/
|   |-- test_api.py
|   |-- test_predictor.py
|-- Dockerfile
|-- docker-compose.yml
|-- requirements.txt
|-- pyproject.toml
"""

print("Estructura recomendada del proyecto:")
print(project_structure)

# Requirements
requirements = """
# requirements.txt
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0
scikit-learn>=1.4.0
joblib>=1.3.0
numpy>=1.26.0
onnxruntime>=1.17.0     # For ONNX inference
python-multipart>=0.0.9 # For file uploads
httpx>=0.27.0           # For async testing
Pillow>=10.0.0          # For image processing
"""
print("Dependencias:")
print(requirements)

## FastAPI: endpoint de prediccion

Vamos a construir una API completa paso a paso. Los componentes clave son:

1. **Pydantic models**: Definen el esquema de request y response
2. **Lifespan**: Carga el modelo al iniciar la app
3. **Endpoints**: POST /predict para predicciones, GET /health para monitoreo
4. **Error handling**: Manejo robusto de errores

In [ ]:
# Complete FastAPI application code
# Save this as app/main.py

fastapi_app_code = '''
"""ML Model Serving API with FastAPI."""

import time
import logging
from contextlib import asynccontextmanager
from typing import Optional

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, field_validator

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ============================================================
# Pydantic models for request/response validation
# ============================================================

class PredictionRequest(BaseModel):
    """Input data for churn prediction."""
    tenure_months: int = Field(..., ge=1, le=72, description="Months as customer")
    monthly_charges: float = Field(..., ge=0, le=500, description="Monthly bill amount")
    total_charges: float = Field(..., ge=0, description="Total amount billed")
    contract_type: str = Field(..., description="Contract type")
    payment_method: str = Field(..., description="Payment method")
    num_support_tickets: int = Field(..., ge=0, description="Number of support tickets")
    num_products: int = Field(..., ge=1, le=10, description="Number of products")
    has_partner: int = Field(..., ge=0, le=1, description="Has partner (0 or 1)")
    age: int = Field(..., ge=18, le=100, description="Customer age")
    satisfaction_score: float = Field(..., ge=1, le=10, description="Satisfaction score")

    @field_validator("contract_type")
    @classmethod
    def validate_contract(cls, v):
        valid = ["month-to-month", "one-year", "two-year"]
        if v not in valid:
            raise ValueError(f"Must be one of: {valid}")
        return v


class PredictionResponse(BaseModel):
    """Prediction output."""
    prediction: str = Field(..., description="CHURN or NO_CHURN")
    probability: float = Field(..., description="Churn probability")
    model_version: str = Field(..., description="Model version")
    latency_ms: float = Field(..., description="Inference latency in ms")


class HealthResponse(BaseModel):
    """Health check response."""
    status: str
    model_loaded: bool
    model_version: str
    uptime_seconds: float


# ============================================================
# Application state and lifespan
# ============================================================

class AppState:
    """Global application state."""
    model = None
    model_version = "v1.0.0"
    start_time = None


app_state = AppState()


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load model on startup, cleanup on shutdown."""
    # Startup
    logger.info("Loading model...")
    try:
        app_state.model = joblib.load("models/churn_pipeline.joblib")
        app_state.start_time = time.time()
        logger.info(f"Model {app_state.model_version} loaded successfully")
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

    yield  # App is running

    # Shutdown
    logger.info("Shutting down, cleaning up resources...")
    app_state.model = None


# ============================================================
# FastAPI application
# ============================================================

app = FastAPI(
    title="Churn Prediction API",
    description="API for predicting customer churn",
    version="1.0.0",
    lifespan=lifespan,
)


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Check if the service is healthy and the model is loaded."""
    return HealthResponse(
        status="healthy" if app_state.model else "unhealthy",
        model_loaded=app_state.model is not None,
        model_version=app_state.model_version,
        uptime_seconds=time.time() - app_state.start_time if app_state.start_time else 0,
    )


@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    """Make a churn prediction for a customer."""
    if app_state.model is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Model not loaded"
        )

    try:
        start = time.time()

        # Convert request to DataFrame (matching training format)
        import pandas as pd
        input_df = pd.DataFrame([request.model_dump()])

        # Predict
        prediction = app_state.model.predict(input_df)[0]
        probability = app_state.model.predict_proba(input_df)[0][1]

        latency = (time.time() - start) * 1000

        return PredictionResponse(
            prediction="CHURN" if prediction == 1 else "NO_CHURN",
            probability=round(float(probability), 4),
            model_version=app_state.model_version,
            latency_ms=round(latency, 2),
        )

    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Prediction error: {str(e)}"
        )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("Codigo completo de la API FastAPI:")
print("=" * 60)
print(fastapi_app_code)
print("\nPara ejecutar: uvicorn app.main:app --reload")
print("Documentacion auto-generada en: http://localhost:8000/docs")

## Ejemplo: servir un modelo de clasificacion

Vamos a crear un ejemplo completo end-to-end: entrenar un modelo simple,
guardarlo, y crear la API que lo sirve.

In [ ]:
# End-to-end example: train, save, and serve a model

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import json

# 1. Generate synthetic data
X, y = make_classification(
    n_samples=1000, n_features=5, n_informative=3,
    n_redundant=1, random_state=42
)
feature_names = ["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]
X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train a pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print("Modelo entrenado:")
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

# 3. Save the model
model_path = "demo_model.joblib"
joblib.dump(pipeline, model_path)
print(f"Modelo guardado en: {model_path}")

# 4. Create FastAPI app that serves this model
serve_code = f'''
"""Simple ML serving example with FastAPI."""

from contextlib import asynccontextmanager
import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List


class Features(BaseModel):
    """Input features for prediction."""
    {chr(10).join(f'    {name}: float = Field(..., description="{name}")' for name in feature_names)}


class Prediction(BaseModel):
    """Prediction response."""
    predicted_class: int
    probability: List[float]


model = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    model = joblib.load("{model_path}")
    yield
    model = None


app = FastAPI(title="Simple ML API", lifespan=lifespan)


@app.post("/predict", response_model=Prediction)
async def predict(features: Features):
    data = np.array([[getattr(features, name) for name in {feature_names}]])
    pred = model.predict(data)[0]
    proba = model.predict_proba(data)[0].tolist()
    return Prediction(predicted_class=int(pred), probability=proba)


@app.get("/health")
async def health():
    return {{"status": "ok", "model_loaded": model is not None}}
'''

print("\nCodigo del servidor:")
print("=" * 60)
print(serve_code)

# 5. Test the model locally (simulating API call)
loaded_model = joblib.load(model_path)
sample = X_test.iloc[0:1]
prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[0].tolist()

print(f"\nTest local:")
print(f"  Input: {sample.to_dict('records')[0]}")
print(f"  Prediction: {prediction}")
print(f"  Probability: {[round(p, 4) for p in probability]}")

## Ejemplo: servir un modelo PyTorch

Servir modelos de deep learning (PyTorch, TensorFlow) requiere consideraciones adicionales:
- **Preprocesamiento de imagenes**: Resize, normalize, convert to tensor
- **Input como base64**: Las imagenes se envian codificadas en base64
- **GPU management**: Mover datos al device correcto
- **Batch inference**: Agrupar requests para eficiencia

In [ ]:
# FastAPI app for serving a PyTorch image classifier

pytorch_app_code = '''
"""PyTorch Image Classification API."""

import base64
import io
import time
from contextlib import asynccontextmanager
from typing import List

import torch
import torchvision.transforms as transforms
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


# ---- Pydantic models ----

class ImageRequest(BaseModel):
    """Request with base64-encoded image."""
    image_base64: str = Field(..., description="Base64 encoded image")
    top_k: int = Field(default=5, ge=1, le=20, description="Number of top predictions")


class ClassPrediction(BaseModel):
    """Single class prediction."""
    class_name: str
    class_id: int
    probability: float


class ImageResponse(BaseModel):
    """Response with predictions."""
    predictions: List[ClassPrediction]
    latency_ms: float


# ---- Model loading and preprocessing ----

# ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ImageNet class labels (abbreviated)
# In production, load from a JSON file
IMAGENET_CLASSES = {}  # Load from imagenet_classes.json

model = None
device = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load PyTorch model on startup."""
    global model, device

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load pretrained ResNet50
    from torchvision.models import resnet50, ResNet50_Weights
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model = model.to(device)
    model.eval()  # Set to evaluation mode

    # Load ImageNet class names
    global IMAGENET_CLASSES
    IMAGENET_CLASSES = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

    print(f"Model loaded on {device}")
    yield
    model = None


app = FastAPI(
    title="Image Classification API",
    version="1.0.0",
    lifespan=lifespan,
)


@app.post("/classify", response_model=ImageResponse)
async def classify_image(request: ImageRequest):
    """Classify an image sent as base64."""
    start = time.time()

    try:
        # Decode base64 image
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image: {e}")

    # Preprocess
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    # Inference (no gradient computation needed)
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Get top-k predictions
    top_probs, top_indices = torch.topk(probabilities, request.top_k)

    predictions = [
        ClassPrediction(
            class_name=IMAGENET_CLASSES[idx.item()],
            class_id=idx.item(),
            probability=round(prob.item(), 4),
        )
        for prob, idx in zip(top_probs, top_indices)
    ]

    latency = (time.time() - start) * 1000
    return ImageResponse(predictions=predictions, latency_ms=round(latency, 2))


@app.get("/health")
async def health():
    return {
        "status": "healthy" if model else "unhealthy",
        "device": str(device),
        "model": "ResNet50",
    }
'''

print("Codigo FastAPI para servir modelo PyTorch (clasificacion de imagenes):")
print("=" * 70)
print(pytorch_app_code)

# Show how to call this API
client_code = '''
# Example client code to call the image classification API
import base64
import requests

# Read and encode image
with open("cat.jpg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# Call API
response = requests.post(
    "http://localhost:8000/classify",
    json={"image_base64": image_b64, "top_k": 5}
)

result = response.json()
for pred in result["predictions"]:
    print(f"{pred['class_name']}: {pred['probability']:.1%}")
'''
print("\nEjemplo de cliente:")
print(client_code)

## Docker para ML

**Docker** permite empaquetar toda la aplicacion (codigo, modelo, dependencias) en un
contenedor reproducible que funciona igual en cualquier maquina.

### Consideraciones para ML:
- Usar **multi-stage builds** para reducir el tamano de la imagen
- Copiar el **modelo** dentro de la imagen o montarlo como volumen
- Configurar **recursos** (memoria, GPU) apropiadamente
- Usar **health checks** para orquestadores (Kubernetes)

In [ ]:
# Dockerfile for ML service
dockerfile_content = '''
# ============================================================
# Dockerfile for ML Model Serving
# ============================================================

# Use Python slim image for smaller size
FROM python:3.11-slim AS base

# Set working directory
WORKDIR /app

# Install system dependencies (if needed for sklearn, numpy, etc.)
RUN apt-get update && \\
    apt-get install -y --no-install-recommends gcc && \\
    rm -rf /var/lib/apt/lists/*

# Copy and install Python dependencies first (for Docker cache)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app/ ./app/

# Copy model artifacts
COPY models/ ./models/

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run the application
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

print("Dockerfile:")
print("=" * 60)
print(dockerfile_content)

# Docker Compose
docker_compose_content = '''
# docker-compose.yml
version: "3.8"

services:
  ml-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/model_v1.joblib
      - LOG_LEVEL=info
      - MAX_BATCH_SIZE=32
    volumes:
      - ./models:/app/models:ro  # Read-only model mount
    deploy:
      resources:
        limits:
          memory: 2G
          cpus: "2.0"
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped

  # Optional: monitoring with Prometheus
  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    depends_on:
      - ml-api
'''

print("\ndocker-compose.yml:")
print("=" * 60)
print(docker_compose_content)

# Build and run commands
print("\nComandos Docker:")
print("  docker build -t ml-api .")
print("  docker run -p 8000:8000 ml-api")
print("  docker-compose up -d")
print("  docker-compose logs -f ml-api")

## Testing del servicio

Testear un servicio ML es tan importante como testear cualquier otro software.
FastAPI proporciona un `TestClient` (basado en httpx) que facilita los tests sin
necesidad de levantar el servidor.

In [ ]:
# Testing FastAPI ML service with TestClient

test_code = '''
"""Tests for ML serving API."""
# tests/test_api.py

import pytest
from fastapi.testclient import TestClient
from app.main import app


client = TestClient(app)


class TestHealthEndpoint:
    """Tests for the health check endpoint."""

    def test_health_returns_200(self):
        response = client.get("/health")
        assert response.status_code == 200

    def test_health_model_loaded(self):
        response = client.get("/health")
        data = response.json()
        assert data["model_loaded"] is True
        assert data["status"] == "healthy"


class TestPredictEndpoint:
    """Tests for the prediction endpoint."""

    @pytest.fixture
    def valid_input(self):
        return {
            "tenure_months": 12,
            "monthly_charges": 65.0,
            "total_charges": 780.0,
            "contract_type": "month-to-month",
            "payment_method": "credit_card",
            "num_support_tickets": 2,
            "num_products": 3,
            "has_partner": 1,
            "age": 35,
            "satisfaction_score": 7.5,
        }

    def test_predict_returns_200(self, valid_input):
        response = client.post("/predict", json=valid_input)
        assert response.status_code == 200

    def test_predict_response_structure(self, valid_input):
        response = client.post("/predict", json=valid_input)
        data = response.json()
        assert "prediction" in data
        assert "probability" in data
        assert "model_version" in data
        assert "latency_ms" in data

    def test_predict_valid_prediction(self, valid_input):
        response = client.post("/predict", json=valid_input)
        data = response.json()
        assert data["prediction"] in ["CHURN", "NO_CHURN"]
        assert 0 <= data["probability"] <= 1

    def test_predict_invalid_contract_type(self, valid_input):
        valid_input["contract_type"] = "invalid"
        response = client.post("/predict", json=valid_input)
        assert response.status_code == 422  # Validation error

    def test_predict_missing_field(self):
        response = client.post("/predict", json={"tenure_months": 12})
        assert response.status_code == 422

    def test_predict_invalid_range(self, valid_input):
        valid_input["tenure_months"] = -5  # Invalid: must be >= 1
        response = client.post("/predict", json=valid_input)
        assert response.status_code == 422

    def test_predict_latency_acceptable(self, valid_input):
        response = client.post("/predict", json=valid_input)
        data = response.json()
        assert data["latency_ms"] < 1000  # Should be under 1 second


class TestBatchPrediction:
    """Tests for batch prediction consistency."""

    def test_same_input_same_output(self):
        """Deterministic predictions."""
        input_data = {
            "tenure_months": 24,
            "monthly_charges": 50.0,
            "total_charges": 1200.0,
            "contract_type": "one-year",
            "payment_method": "bank_transfer",
            "num_support_tickets": 1,
            "num_products": 4,
            "has_partner": 1,
            "age": 45,
            "satisfaction_score": 8.0,
        }
        r1 = client.post("/predict", json=input_data).json()
        r2 = client.post("/predict", json=input_data).json()
        assert r1["prediction"] == r2["prediction"]
        assert r1["probability"] == r2["probability"]
'''

print("Codigo de tests:")
print("=" * 60)
print(test_code)

print("\nEjecutar tests:")
print("  pytest tests/ -v")
print("  pytest tests/ -v --cov=app  # With coverage")

## Optimizacion: ONNX Runtime

**ONNX Runtime** es un motor de inferencia de alto rendimiento que puede ejecutar modelos
exportados al formato ONNX con optimizaciones significativas de velocidad.

### Ventajas de servir con ONNX:
- **2-10x mas rapido** que PyTorch/sklearn en CPU
- **Menor uso de memoria**: Optimizaciones de grafo
- **Sin dependencia de PyTorch**: Imagen Docker mas pequena
- **Cuantizacion**: Reducir precision (FP32 -> INT8) para mas velocidad

In [ ]:
# Convert sklearn model to ONNX and benchmark
# !pip install skl2onnx onnxruntime

import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# Train a model for benchmarking
X_bench, y_bench = make_classification(
    n_samples=5000, n_features=20, random_state=42
)

sklearn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
])
sklearn_pipeline.fit(X_bench[:4000], y_bench[:4000])
X_test_bench = X_bench[4000:].astype(np.float32)

print("Modelo sklearn entrenado para benchmark.")

# Convert to ONNX
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnxruntime as ort

    # Define input type
    initial_type = [("float_input", FloatTensorType([None, X_bench.shape[1]]))]

    # Convert to ONNX
    onnx_model = convert_sklearn(
        sklearn_pipeline,
        initial_types=initial_type,
        target_opset=12,
    )

    # Save ONNX model
    onnx_path = "model_benchmark.onnx"
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f"Modelo ONNX guardado: {onnx_path}")

    # Create ONNX Runtime session
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name

    # Verify predictions match
    sklearn_preds = sklearn_pipeline.predict(X_test_bench[:5])
    onnx_preds = session.run(None, {input_name: X_test_bench[:5]})[0]
    print(f"\nPredicciones coinciden: {np.array_equal(sklearn_preds, onnx_preds)}")

    # Benchmark: sklearn vs ONNX Runtime
    n_iterations = 100

    # Single sample inference (most common in API)
    single_sample = X_test_bench[:1]

    # Warmup
    for _ in range(10):
        sklearn_pipeline.predict(single_sample)
        session.run(None, {input_name: single_sample})

    # Benchmark sklearn
    start = time.time()
    for _ in range(n_iterations):
        sklearn_pipeline.predict(single_sample)
    sklearn_time = (time.time() - start) / n_iterations * 1000

    # Benchmark ONNX Runtime
    start = time.time()
    for _ in range(n_iterations):
        session.run(None, {input_name: single_sample})
    onnx_time = (time.time() - start) / n_iterations * 1000

    # Batch inference benchmark
    batch = X_test_bench[:100]

    start = time.time()
    for _ in range(n_iterations):
        sklearn_pipeline.predict(batch)
    sklearn_batch_time = (time.time() - start) / n_iterations * 1000

    start = time.time()
    for _ in range(n_iterations):
        session.run(None, {input_name: batch})
    onnx_batch_time = (time.time() - start) / n_iterations * 1000

    print(f"\n{'=' * 55}")
    print(f"BENCHMARK: sklearn vs ONNX Runtime")
    print(f"{'=' * 55}")
    print(f"{'Escenario':<25} {'sklearn':>10} {'ONNX':>10} {'Speedup':>10}")
    print(f"{'-' * 55}")
    print(f"{'Single sample (ms)':<25} {sklearn_time:>10.3f} {onnx_time:>10.3f} {sklearn_time/onnx_time:>10.1f}x")
    print(f"{'Batch 100 (ms)':<25} {sklearn_batch_time:>10.3f} {onnx_batch_time:>10.3f} {sklearn_batch_time/onnx_batch_time:>10.1f}x")

    # File size comparison
    import os
    joblib.dump(sklearn_pipeline, "model_benchmark.joblib")
    sklearn_size = os.path.getsize("model_benchmark.joblib") / (1024 * 1024)
    onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"\n{'Tamano sklearn (MB)':<25} {sklearn_size:>10.2f}")
    print(f"{'Tamano ONNX (MB)':<25} {onnx_size:>10.2f}")

except ImportError as e:
    print(f"Instala las dependencias: pip install skl2onnx onnxruntime")
    print(f"Error: {e}")

## Resumen: checklist de deployment

### Antes del deployment
- [ ] Modelo entrenado y evaluado offline
- [ ] Pipeline de preprocesamiento encapsulado (no depende de variables globales)
- [ ] Modelo serializado (joblib/ONNX/TorchScript)
- [ ] Tests unitarios del pipeline de prediccion
- [ ] Benchmarks de latencia y throughput

### API y servicio
- [ ] FastAPI con Pydantic models para validacion
- [ ] Endpoint `/health` para monitoreo
- [ ] Endpoint `/predict` con error handling
- [ ] Modelo cargado en startup (lifespan), no en cada request
- [ ] Logging estructurado (request id, latencia, errores)
- [ ] Versionado del modelo en las respuestas

### Containerizacion
- [ ] Dockerfile optimizado (multi-stage, cache de dependencias)
- [ ] docker-compose para desarrollo local
- [ ] Variables de entorno para configuracion (no hardcoded)
- [ ] Health checks configurados

### Optimizacion
- [ ] Considerar ONNX Runtime para inferencia mas rapida
- [ ] Profiling de latencia (donde se gasta el tiempo?)
- [ ] Batch inference si el caso de uso lo permite
- [ ] Caching de predicciones frecuentes (Redis)
- [ ] Cuantizacion si la precision lo permite

### Monitoreo en produccion
- [ ] Metricas: latencia p50/p95/p99, throughput, errores
- [ ] Data drift detection
- [ ] Model performance monitoring (si hay labels disponibles)
- [ ] Alertas por degradacion de latencia o precision
- [ ] Logs centralizados (ELK, CloudWatch, etc.)

### Patrones avanzados

| Patron | Cuando usarlo |
|--------|---------------|
| **Shadow deployment** | Probar modelo nuevo sin impactar usuarios |
| **A/B testing** | Comparar modelos en produccion |
| **Canary release** | Desplegar gradualmente (1% -> 10% -> 100%) |
| **Blue-green** | Zero-downtime deployment |
| **Feature store** | Reutilizar features entre modelos |
| **Model registry** | Versionado y linaje de modelos (MLflow) |